# Exploring and cleaning the book data

Download the book metadata, check missing values, and save the rows used by the other notebooks. Run this notebook from the repository root. The download uses KaggleHub.


In [ ]:
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

path = Path(kagglehub.dataset_download("dylanjcastillo/7k-books-with-metadata"))
books = pd.read_csv(path / "books.csv")
books.head()

## Check missing values


In [ ]:
books.info()

plt.figure(figsize=(12, 5))
sns.heatmap(books.isna().transpose(), cbar=False)
plt.xlabel("Books")
plt.ylabel("Columns")
plt.show()

## Look at the numeric columns

Use 2024 as the reference year to keep the original analysis consistent.


In [ ]:
books["missing_description"] = books["description"].isna().astype(int)
books["age_of_book"] = 2024 - books["published_year"]

columns = ["num_pages", "age_of_book", "missing_description", "average_rating"]
correlations = books[columns].corr(method="spearman")

plt.figure(figsize=(8, 6))
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation between book details")
plt.show()

## Keep books with enough information

Remove missing descriptions, page counts, ratings, and publication years. Keep descriptions with at least 25 words, as in the original notebook.


In [ ]:
required_columns = ["description", "num_pages", "average_rating", "published_year"]
clean_books = books.dropna(subset=required_columns).copy()
clean_books["words_in_description"] = clean_books["description"].str.split().str.len()

clean_books["words_in_description"].describe()

In [ ]:
clean_books = clean_books.loc[clean_books["words_in_description"] >= 25].copy()
clean_books["categories"].value_counts().head(15)

## Add display titles and tagged descriptions


In [ ]:
clean_books["title_and_subtitle"] = clean_books["title"]
has_subtitle = clean_books["subtitle"].notna()
clean_books.loc[has_subtitle, "title_and_subtitle"] = (
    clean_books.loc[has_subtitle, "title"]
    + ": "
    + clean_books.loc[has_subtitle, "subtitle"]
)

# The ISBN lets us match a search result back to its book.
clean_books["tagged_description"] = (
    clean_books["isbn13"].astype(str) + " " + clean_books["description"]
)
clean_books[["title_and_subtitle", "tagged_description"]].head()

## Save the cleaned data


In [ ]:
temporary_columns = [
    "subtitle",
    "missing_description",
    "age_of_book",
    "words_in_description",
]
clean_books = clean_books.drop(columns=temporary_columns)
clean_books.to_csv("books_cleaned.csv", index=False)
print(f"Saved {len(clean_books)} books to books_cleaned.csv")